#https://data.cityofnewyork.us/Environment/Energy-and-Water-Data-Disclosure-for-Local-Law-84-/7x5e-2fxh/about_data


In [1]:
# Import necessary libraries for data processing and spatial operations.
import os  # For handling file paths.
import pandas as pd  # For handling tabular data (CSV files).
import numpy as np  # For numerical operations and array manipulations.
from scipy.spatial import cKDTree  # For efficient spatial queries using KD-tree.
import tqdm  # Import the tqdm module for accessing its version.
from tqdm import tqdm as tqdm_progress  # Alias the tqdm function for progress bars.
import scipy  # Import scipy to retrieve its version.
# Import sys to get the Python version.
import sys

# Debug: Confirm that imports are successful.
print("Debug: Libraries imported successfully.")

# Print the version of each imported library.
print(f"Debug: pandas version: {pd.__version__}")
print(f"Debug: numpy version: {np.__version__}")
print(f"Debug: scipy version: {scipy.__version__}")
print(f"Debug: tqdm version: {tqdm.__version__}")
# Note: os is part of Python's standard library and does not have a separate version.
# Its functionality is tied to the Python version printed above.

Debug: Libraries imported successfully.
Debug: pandas version: 2.2.3
Debug: numpy version: 1.26.4
Debug: scipy version: 1.13.1
Debug: tqdm version: 4.67.1


In [2]:
# Define directory paths for Kaggle environment.
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets.
sub_dir = r"/kaggle/working/"  # Submission directory for output files.

# Define file paths for input datasets.
train_file = f"{base_dir}/Training_data.csv"
valid_file = f"{base_dir}/Validation_data.csv"
energy_file = f"{base_dir}/Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250224.csv"
#https://data.cityofnewyork.us/Environment/Energy-and-Water-Data-Disclosure-for-Local-Law-84-/7x5e-2fxh/about_data


# Define output file paths for enriched datasets.
output_train_csv = f"{sub_dir}/training_data_ENERGY_DISCLOSURE.csv"
output_valid_csv = f"{sub_dir}/validation_data_ENERGY_DISCLOSURE.csv"

# Debug: Print the file paths to confirm they are set correctly.
print(f"Debug: Training file path: {train_file}")
print(f"Debug: Validation file path: {valid_file}")
print(f"Debug: Energy disclosure file path: {energy_file}")
print(f"Debug: Output training CSV path: {output_train_csv}")
print(f"Debug: Output validation CSV path: {output_valid_csv}")

Debug: Training file path: /kaggle/input/eyds-base-dataset/Training_data.csv
Debug: Validation file path: /kaggle/input/eyds-base-dataset/Validation_data.csv
Debug: Energy disclosure file path: /kaggle/input/eyds-base-dataset/Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250224.csv
Debug: Output training CSV path: /kaggle/working//training_data_ENERGY_DISCLOSURE.csv
Debug: Output validation CSV path: /kaggle/working//validation_data_ENERGY_DISCLOSURE.csv


In [3]:
def calculate_energy_features_vectorized(locations_df, energy_df, radii_meters):
    """
    For each location (with 'Latitude' and 'Longitude'), calculates the sum of 
    selected columns from the Energy and Water Data Disclosure dataset 
    for all properties within each specified radius (in meters).

    Parameters:
      locations_df (pd.DataFrame): DataFrame containing at least ['Latitude', 'Longitude']
      energy_df (pd.DataFrame): DataFrame with columns at least:
                     ['Latitude', 'Longitude', 'Net Emissions (Metric Tons CO2e)', 
                      'Weather Normalized Site Energy Use (kBtu)', 
                      'Weather Normalized Site Natural Gas Use (therms)',
                      'Weather Normalized Site Electricity (kWh)', ... ]
      radii_meters (list): List of radii in meters (e.g., [500, 1000])

    Returns:
      pd.DataFrame: locations_df with new columns:
        sum_net_emissions_{r}m, sum_site_energy_{r}m, sum_natural_gas_{r}m, sum_electricity_{r}m
    """
    # Debug: Print the shapes of input DataFrames.
    print(f"Debug: Locations DataFrame shape: {locations_df.shape}")
    print(f"Debug: Energy DataFrame shape: {energy_df.shape}")

    # 1. Extract coordinates from the main DataFrame and convert to radians.
    loc_coords = locations_df[['Latitude', 'Longitude']].values
    loc_coords_rad = np.radians(loc_coords)

    # Debug: Print the number of locations being processed.
    print(f"Debug: Number of locations to process: {len(loc_coords)}")

    # 2. Extract coordinates from the energy DataFrame and build a KD-tree.
    energy_coords = energy_df[['Latitude', 'Longitude']].values
    energy_coords_rad = np.radians(energy_coords)
    energy_kdtree = cKDTree(energy_coords_rad)

    # Debug: Confirm that the KD-tree was created.
    print("Debug: KD-tree for energy data created successfully.")

    # 3. Precompute each radius in radians (Earth radius ~ 6,371,000 m).
    radius_radians = {r: r / 6371000.0 for r in radii_meters}

    # Debug: Print the radii being used.
    print(f"Debug: Radii (in meters) for feature computation: {radii_meters}")

    # 4. Initialize arrays to store sums for each radius.
    result_sums = {}
    columns_to_sum = [
        'Net Emissions (Metric Tons CO2e)',
        'Weather Normalized Site Energy Use (kBtu)',
        'Weather Normalized Site Natural Gas Use (therms)',
        'Weather Normalized Site Electricity (kWh)'
    ]

    # Create a zero array for each (radius, column) pair.
    for r in radii_meters:
        for col in columns_to_sum:
            result_sums[(r, col)] = np.zeros(len(locations_df), dtype=float)

    # 5. Process the locations in batches for efficiency.
    batch_size = 1000
    num_batches = int(np.ceil(len(locations_df) / batch_size))

    # Debug: Print the number of batches to process.
    print(f"Debug: Number of batches to process: {num_batches}")

    for i in tqdm_progress(range(num_batches), desc="Processing energy features"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(locations_df))
        batch_coords_rad = loc_coords_rad[start_idx:end_idx]
        
        # For each specified radius, query the KD-tree.
        for r in radii_meters:
            r_rad = radius_radians[r]
            indices_list = energy_kdtree.query_ball_point(batch_coords_rad, r=r_rad, workers=-1)
            
            # For each location in the batch, sum the values in the specified columns.
            for j, indices in enumerate(indices_list):
                if not indices:
                    continue
                for col in columns_to_sum:
                    sum_val = energy_df.iloc[indices][col].sum(skipna=True)
                    result_sums[(r, col)][start_idx + j] = sum_val

    # 6. Append new columns with the computed sums to locations_df.
    for r in radii_meters:
        for col in columns_to_sum:
            # Create a consistent column name. For example, 'Net Emissions (Metric Tons CO2e)' becomes
            # 'sum_net_emissions_mtco2e_500m'
            base_col_name = col.lower().replace(' ', '_').replace('(metric_tons_co2e)', 'mtco2e')
            new_col = f"sum_{base_col_name}_{r}m"
            locations_df[new_col] = result_sums[(r, col)]

    # Debug: Print the new columns added to the DataFrame.
    new_cols = [col for col in locations_df.columns if col.startswith('sum_')]
    print(f"Debug: New columns added to locations DataFrame: {new_cols}")

    return locations_df

In [4]:
def main():
    """
    Main function to calculate energy disclosure features (sums of energy-related metrics) for training and validation datasets
    and save the augmented data to new CSV files.
    """
    # Define the radii (in meters) for which to calculate the sums.
    radii = [500, 1000]  # For example, 500m and 1000m

    # Debug: Print the radii being used.
    print(f"Debug: Radii for energy features: {radii}")

    # 1. Read the Energy and Water Data Disclosure data.
    print("Reading energy disclosure data...")
    try:
        energy_df = pd.read_csv(energy_file)
    except FileNotFoundError:
        print(f"Error: File not found at {energy_file}")
        return

    # Debug: Print the shape and columns of the energy disclosure data.
    print(f"Debug: Energy DataFrame shape: {energy_df.shape}")
    print(f"Debug: Energy DataFrame columns: {energy_df.columns.tolist()}")

    # Define the required columns.
    required_energy_cols = [
        'Latitude', 
        'Longitude', 
        'Net Emissions (Metric Tons CO2e)', 
        'Weather Normalized Site Energy Use (kBtu)',
        'Weather Normalized Site Natural Gas Use (therms)',
        'Weather Normalized Site Electricity (kWh)'
    ]
    missing_cols = [c for c in required_energy_cols if c not in energy_df.columns]
    if missing_cols:
        print("Error: The following required columns are missing in energy_df:", missing_cols)
        return

    # Debug: Confirm that required columns are present.
    print("Debug: Required columns verified in energy DataFrame.")

    # Convert 'Latitude' and 'Longitude' to numeric.
    energy_df['Latitude'] = pd.to_numeric(energy_df['Latitude'], errors='coerce')
    energy_df['Longitude'] = pd.to_numeric(energy_df['Longitude'], errors='coerce')
    
    # Convert each column to sum (except lat/lon) to numeric.
    for col in required_energy_cols:
        if col not in ['Latitude', 'Longitude']:
            energy_df[col] = pd.to_numeric(energy_df[col], errors='coerce')
    
    # Drop rows where Latitude or Longitude is NaN.
    energy_df.dropna(subset=['Latitude', 'Longitude'], inplace=True)

    # Debug: Print the shape after dropping NaN values.
    print(f"Debug: Energy DataFrame shape after dropping NaN lat/lon: {energy_df.shape}")

    # 2. Read training and validation data.
    print("Reading training data...")
    try:
        train_data = pd.read_csv(train_file)
    except FileNotFoundError:
        print(f"Error: Training data file not found at {train_file}")
        return

    # Debug: Print the shape and columns of the training data.
    print(f"Debug: Training DataFrame shape: {train_data.shape}")
    print(f"Debug: Training DataFrame columns: {train_data.columns.tolist()}")

    print("Reading validation data...")
    try:
        validation_data = pd.read_csv(valid_file)
    except FileNotFoundError:
        print(f"Error: Validation data file not found at {valid_file}")
        return

    # Debug: Print the shape and columns of the validation data.
    print(f"Debug: Validation DataFrame shape: {validation_data.shape}")
    print(f"Debug: Validation DataFrame columns: {validation_data.columns.tolist()}")

    # 3. Ensure that both training and validation data have 'Latitude' and 'Longitude'.
    for df, name in [(train_data, "Training"), (validation_data, "Validation")]:
        if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
            print(f"Error: 'Latitude' or 'Longitude' column missing in {name} dataset.")
            return

    # Debug: Confirm that latitude and longitude columns are present.
    print("Debug: Latitude and Longitude columns verified in both datasets.")

    # 4. Compute energy sums for training data.
    print("Calculating energy sums for training data...")
    train_data_aug = calculate_energy_features_vectorized(
        locations_df=train_data.copy(),
        energy_df=energy_df,
        radii_meters=radii
    )

    # Debug: Print the shape of the training data after feature calculation.
    print(f"Debug: Training DataFrame shape after feature calculation: {train_data_aug.shape}")

    # 5. Compute energy sums for validation data.
    print("Calculating energy sums for validation data...")
    validation_data_aug = calculate_energy_features_vectorized(
        locations_df=validation_data.copy(),
        energy_df=energy_df,
        radii_meters=radii
    )

    # Debug: Print the shape of the validation data after feature calculation.
    print(f"Debug: Validation DataFrame shape after feature calculation: {validation_data_aug.shape}")

    # 6. Optionally drop columns not needed (e.g., 'geometry' or 'datetime').
    cols_to_drop = ['geometry', 'datetime']
    train_data_aug.drop(columns=cols_to_drop, errors='ignore', inplace=True)
    validation_data_aug.drop(columns=cols_to_drop, errors='ignore', inplace=True)

    # Debug: Print the columns after dropping.
    print(f"Debug: Training DataFrame columns after dropping: {train_data_aug.columns.tolist()}")
    print(f"Debug: Validation DataFrame columns after dropping: {validation_data_aug.columns.tolist()}")

    # 7. Save the augmented DataFrames to new CSV files.
    print("Saving results...")
    train_data_aug.to_csv(output_train_csv, index=False)
    validation_data_aug.to_csv(output_valid_csv, index=False)

    # Debug: Print the final confirmation messages with file paths.
    print(f"Debug: Augmented training data saved to: {output_train_csv}")
    print(f"Debug: Augmented validation data saved to: {output_valid_csv}")

    print("All done!")

if __name__ == "__main__":
    main()

Debug: Radii for energy features: [500, 1000]
Reading energy disclosure data...


<ipython-input-4-4c866f2195da>:15: DtypeWarning: Columns (9,15,216,217) have mixed types. Specify dtype option on import or set low_memory=False.
  energy_df = pd.read_csv(energy_file)


Debug: Energy DataFrame shape: (29842, 249)
Debug: Energy DataFrame columns: ['Property Id', 'Property Name', 'Parent Property Id', 'Parent Property Name', 'Year Ending', 'NYC Borough, Block and Lot (BBL)', 'NYC Building Identification Number (BIN)', 'Address 1', 'City', 'Postal Code', 'Primary Property Type - Self Selected', 'Primary Property Type - Portfolio Manager-Calculated', 'National Median Reference Property Type', 'List of All Property Use Types at Property', 'Largest Property Use Type', 'Largest Property Use Type - Gross Floor Area (ft²)', '2nd Largest Property Use Type', '2nd Largest Property Use - Gross Floor Area (ft²)', '3rd Largest Property Use Type', '3rd Largest Property Use Type - Gross Floor Area (ft²)', 'Year Built', 'Construction Status', 'Number of Buildings', 'Occupancy', 'Metered Areas (Energy)', 'Metered Areas (Water)', 'ENERGY STAR Score', 'National Median ENERGY STAR Score', 'Target ENERGY STAR Score', 'Reason(s) for No Score', 'ENERGY STAR Certification - Ye

Processing energy features: 100%|██████████| 12/12 [06:43<00:00, 33.65s/it]


Debug: New columns added to locations DataFrame: ['sum_net_emissions_mtco2e_500m', 'sum_weather_normalized_site_energy_use_(kbtu)_500m', 'sum_weather_normalized_site_natural_gas_use_(therms)_500m', 'sum_weather_normalized_site_electricity_(kwh)_500m', 'sum_net_emissions_mtco2e_1000m', 'sum_weather_normalized_site_energy_use_(kbtu)_1000m', 'sum_weather_normalized_site_natural_gas_use_(therms)_1000m', 'sum_weather_normalized_site_electricity_(kwh)_1000m']
Debug: Training DataFrame shape after feature calculation: (11229, 12)
Calculating energy sums for validation data...
Debug: Locations DataFrame shape: (1040, 3)
Debug: Energy DataFrame shape: (28713, 249)
Debug: Number of locations to process: 1040
Debug: KD-tree for energy data created successfully.
Debug: Radii (in meters) for feature computation: [500, 1000]
Debug: Number of batches to process: 2


Processing energy features: 100%|██████████| 2/2 [00:40<00:00, 20.18s/it]


Debug: New columns added to locations DataFrame: ['sum_net_emissions_mtco2e_500m', 'sum_weather_normalized_site_energy_use_(kbtu)_500m', 'sum_weather_normalized_site_natural_gas_use_(therms)_500m', 'sum_weather_normalized_site_electricity_(kwh)_500m', 'sum_net_emissions_mtco2e_1000m', 'sum_weather_normalized_site_energy_use_(kbtu)_1000m', 'sum_weather_normalized_site_natural_gas_use_(therms)_1000m', 'sum_weather_normalized_site_electricity_(kwh)_1000m']
Debug: Validation DataFrame shape after feature calculation: (1040, 11)
Debug: Training DataFrame columns after dropping: ['Longitude', 'Latitude', 'UHI Index', 'sum_net_emissions_mtco2e_500m', 'sum_weather_normalized_site_energy_use_(kbtu)_500m', 'sum_weather_normalized_site_natural_gas_use_(therms)_500m', 'sum_weather_normalized_site_electricity_(kwh)_500m', 'sum_net_emissions_mtco2e_1000m', 'sum_weather_normalized_site_energy_use_(kbtu)_1000m', 'sum_weather_normalized_site_natural_gas_use_(therms)_1000m', 'sum_weather_normalized_site